# Notebook 06: Ad Click Prediction

## Why This Matters

CTR prediction is one of the highest-value ML problems in industry. Google generates $280B/year in ad revenue; Meta ~$115B; the entire digital advertising market is ~$600B. A 0.1% improvement in CTR prediction accuracy can be worth hundreds of millions of dollars in auction efficiency. Every ad platform runs sophisticated ML for CTR prediction. Understanding this system is essential for advertising, marketplace, and platform ML roles.

In [ ]:
%pip install -q torch numpy pandas scikit-learn matplotlib

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, log_loss
torch.manual_seed(42); np.random.seed(42)
print("Ready.")

## 1. The Ad Auction: Why CTR Prediction Matters

In a second-price auction (Google, Meta), the winner is selected by:
`rank_score = bid x predicted_CTR`

This means CTR prediction directly determines:
1. Who wins the auction
2. How much the winner pays (second price)
3. Revenue to the platform

**Critical implication**: CTR prediction must be **calibrated**. If the model overestimates CTR, advertisers overbid, ROI drops, they reduce bids, platform revenue falls. The auction mechanism assumes the model tells the truth about click probability.

In [ ]:
np.random.seed(42)
N_ADS = 10
ads = pd.DataFrame({
    "bid": np.random.uniform(0.5, 5.0, N_ADS),
    "true_ctr": np.random.beta(1, 20, N_ADS),
})
ads["calibrated_ctr"] = ads.true_ctr * np.random.uniform(0.9, 1.1, N_ADS)
ads["inflated_ctr"] = ads.true_ctr * np.random.uniform(2.5, 3.5, N_ADS)

def run_auction(ads, ctr_col, n_auctions=10000):
    total_revenue = 0
    for _ in range(n_auctions):
        ads_copy = ads.copy()
        ads_copy["rank_score"] = ads_copy.bid * ads_copy[ctr_col]
        sorted_ads = ads_copy.sort_values("rank_score", ascending=False).reset_index(drop=True)
        winner = sorted_ads.iloc[0]; second = sorted_ads.iloc[1]
        cpc = second.rank_score / winner[ctr_col]
        clicked = np.random.rand() < winner.true_ctr
        total_revenue += cpc if clicked else 0
    return total_revenue / n_auctions

rev_cal = run_auction(ads, "calibrated_ctr")
rev_inf = run_auction(ads, "inflated_ctr")
print("=== Auction Revenue: Calibrated vs Inflated CTR ===")
print(f"Calibrated model revenue: ${rev_cal:.4f}/auction")
print(f"Inflated model revenue:   ${rev_inf:.4f}/auction")
print(f"Revenue impact: {(rev_cal-rev_inf)/rev_cal:.1%}")
print("\nInflated CTR -> overbidding -> advertiser ROI collapses -> bids fall -> platform revenue falls")

## 2. Wide & Deep Architecture

Facebook's (2016) and Google's Wide & Deep papers both addressed a key tension:
- **Memorization (wide)**: logistic regression on feature crosses learns exact patterns ('user from NYC + sports ad -> high CTR')
- **Generalization (deep)**: neural network learns dense embeddings that generalize to unseen combinations

Wide & Deep combines both:
- **Wide path**: logistic regression on raw + crossed sparse features
- **Deep path**: embeddings -> MLP
- **Combined**: outputs concatenated before final sigmoid

**Why not just deep?** Deep models need lots of data to learn rare feature combinations. Wide models memorize them from training set directly.

In [ ]:
class WideAndDeep(nn.Module):
    def __init__(self, n_users, n_ads, n_wide, n_deep, embed_dim=32):
        super().__init__()
        self.user_embed = nn.Embedding(n_users, embed_dim)
        self.ad_embed = nn.Embedding(n_ads, embed_dim)
        self.deep = nn.Sequential(
            nn.Linear(embed_dim*2 + n_deep, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU()
        )
        self.wide = nn.Linear(n_wide, 1, bias=False)
        self.output = nn.Linear(64 + 1, 1)
    
    def forward(self, user_ids, ad_ids, wide_feats, deep_feats):
        wide_out = self.wide(wide_feats)
        u = self.user_embed(user_ids); a = self.ad_embed(ad_ids)
        deep_out = self.deep(torch.cat([u, a, deep_feats], dim=-1))
        return torch.sigmoid(self.output(torch.cat([wide_out, deep_out], dim=-1))).squeeze(-1)

N_USERS, N_ADS_SIM, N_WIDE, N_DEEP, N_SAMPLES = 1000, 500, 50, 20, 20000
user_ids = torch.randint(0, N_USERS, (N_SAMPLES,))
ad_ids = torch.randint(0, N_ADS_SIM, (N_SAMPLES,))
wide_feats = torch.randn(N_SAMPLES, N_WIDE)
deep_feats = torch.randn(N_SAMPLES, N_DEEP)
true_ctr_logit = 0.5*wide_feats[:,0] + 0.3*deep_feats[:,0]*deep_feats[:,1] - 3.0
labels = torch.bernoulli(torch.sigmoid(true_ctr_logit))

model = WideAndDeep(N_USERS, N_ADS_SIM, N_WIDE, N_DEEP)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()
losses = []
for epoch in range(30):
    model.train()
    for i in range(0, N_SAMPLES, 256):
        u=user_ids[i:i+256]; a=ad_ids[i:i+256]; w=wide_feats[i:i+256]; d=deep_feats[i:i+256]; l=labels[i:i+256]
        optimizer.zero_grad()
        preds = model(u, a, w, d)
        loss = criterion(preds, l)
        loss.backward(); optimizer.step()
    with torch.no_grad(): losses.append(criterion(model(user_ids, ad_ids, wide_feats, deep_feats), labels).item())
model.eval()
with torch.no_grad(): final_preds = model(user_ids, ad_ids, wide_feats, deep_feats).numpy()
print(f"Final BCE loss: {losses[-1]:.4f}")
print(f"Mean predicted CTR: {final_preds.mean():.4f} (true: ~5%)")

## 3. Calibration and Platt Scaling

**Reliability diagram**: bin predictions into 10 buckets. Compare mean(predicted_prob) vs mean(actual_label). Perfect calibration = diagonal line.

**Expected Calibration Error (ECE)**: weighted average of |predicted - actual| across bins.

**Platt scaling**: fit logistic regression on raw model logits. Two parameters (a, b) rescale the output. Simple, effective for fixing systematic over/under-confidence.

**Isotonic regression**: non-parametric, monotonic calibration. More flexible than Platt but needs more data.

In [ ]:
labels_np = labels.numpy(); preds_np = final_preds

logits = np.log(preds_np / (1 - preds_np + 1e-8))
platt = LogisticRegression(max_iter=500).fit(logits.reshape(-1,1), labels_np)
platt_cal = platt.predict_proba(logits.reshape(-1,1))[:,1]

iso = IsotonicRegression(out_of_bounds="clip").fit(preds_np, labels_np)
iso_cal = iso.predict(preds_np)

def ece(y_true, y_pred, n_bins=10):
    bins = np.linspace(0, 1, n_bins+1)
    ece_val = 0
    for b in range(n_bins):
        mask = (y_pred >= bins[b]) & (y_pred < bins[b+1])
        if mask.sum() > 0: ece_val += abs(y_true[mask].mean() - y_pred[mask].mean()) * mask.sum() / len(y_true)
    return ece_val

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for preds, name, color in [(preds_np,"Uncalibrated","blue"),(platt_cal,"Platt","green"),(iso_cal,"Isotonic","orange")]:
    fp, mp = calibration_curve(labels_np, preds, n_bins=10)
    axes[0].plot(mp, fp, "s-", label=name, color=color)
axes[0].plot([0,1],[0,1],"k--",label="Perfect"); axes[0].set_xlabel("Mean Predicted"); axes[0].set_ylabel("Fraction Positive"); axes[0].set_title("Reliability Diagram"); axes[0].legend()
for name, preds in [("Uncalibrated",preds_np),("Platt",platt_cal),("Isotonic",iso_cal)]:
    print(f"{name:20} ECE={ece(labels_np,preds):.4f}")
aces = {"Uncal": ece(labels_np,preds_np), "Platt": ece(labels_np,platt_cal), "Iso": ece(labels_np,iso_cal)}
axes[1].bar(list(aces.keys()), list(aces.values()), color=["blue","green","orange"])
axes[1].set_title("ECE (lower=better)"); axes[1].set_ylabel("ECE")
plt.tight_layout(); plt.savefig("/tmp/ctr_cal.png", dpi=80); plt.show()

## 4. Delayed Feedback Problem

Conversions happen hours to days after clicks. This creates a fundamental training challenge:
- At training time, recent impressions have incomplete conversion labels
- Including them with label=0 causes the model to underestimate conversion rate
- Excluding them wastes recent data

**Solutions**:
1. **Delay cutoff**: only use impressions older than 7 days. Simple but wastes fresh data.
2. **Importance sampling**: weight recent examples by probability of having complete labels.
3. **Hierarchical model**: separate CTR model (label in minutes) from CVR model (label in days).

**Interview tip**: Always mention delayed feedback when designing ad models.

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| CTR feeds auction | Expected value = bid x CTR; miscalibrated CTR breaks auction economics |
| Wide path | LR on feature crosses - memorizes known patterns |
| Deep path | Embeddings + MLP - generalizes to unseen combinations |
| Calibration | Predicted P(click) must match actual frequency |
| Platt scaling | Logistic regression on logits; fixes systematic miscalibration |
| Delayed feedback | Conversion labels arrive days late; naive training underestimates CVR |

### Common Pitfalls
- Optimizing only AUC without checking calibration
- Ignoring the delayed feedback problem for CVR models
- Not normalizing features before the deep path
